# Yahoo Fantasy Baseball

Uses the `Yahoo` class from `yahoo.py` to manage your fantasy team via the Yahoo Fantasy Sports API.

**Credentials required in `.env`:**
```
Client_ID:      "your_yahoo_app_client_id"
Client_Secret:  "your_yahoo_app_client_secret"
Yahoo_Email:    "your_yahoo_account_email"
Yahoo_Password: "your_yahoo_account_password"
```

Section 2 uses Playwright to automate the OAuth browser flow — no manual steps needed.
On subsequent runs, saved tokens are reused automatically.

## Section 1: Setup — load credentials from .env

In [60]:
import json
import logging
import os
import re

import pandas as pd
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)

logging.getLogger('yahoo_oauth').setLevel(logging.WARNING)

# ── Parse .env (Key: "Value" format) ─────────────────────────────────────────
def _parse_env(path='.env'):
    out = {}
    with open(path) as f:
        for line in f:
            m = re.match(r'([\w]+):\s*"?([^"\n]+)"?', line.strip())
            if m:
                out[m.group(1)] = m.group(2).strip()
    return out

env = _parse_env('.env')
CONSUMER_KEY    = env['Client_ID']
CONSUMER_SECRET = env['Client_Secret']
YAHOO_EMAIL     = env.get('Yahoo_Email', '')
YAHOO_PASSWORD  = env.get('Yahoo_Password', '')

print(f'Loaded credentials from .env')
print(f'  Client_ID     : {CONSUMER_KEY[:12]}...')
print(f'  Client_Secret : {CONSUMER_SECRET[:8]}...')
print(f'  Yahoo_Email   : {YAHOO_EMAIL or "(not set — add to .env)"}')
print(f'  Yahoo_Password: {"(set)" if YAHOO_PASSWORD else "(not set — add to .env)"}')

# ── Write browser/yahoo_oauth.json (consumer key/secret only — tokens added after auth) ──
CREDS_FILE = 'browser/yahoo_oauth.json'
os.makedirs('browser', exist_ok=True)

if not os.path.exists(CREDS_FILE):
    with open(CREDS_FILE, 'w') as f:
        json.dump({'consumer_key': CONSUMER_KEY, 'consumer_secret': CONSUMER_SECRET}, f, indent=2)
    print(f'\nCreated {CREDS_FILE}')
else:
    with open(CREDS_FILE) as f:
        existing = json.load(f)
    existing.update({'consumer_key': CONSUMER_KEY, 'consumer_secret': CONSUMER_SECRET})
    with open(CREDS_FILE, 'w') as f:
        json.dump(existing, f, indent=2)
    has_token = 'access_token' in existing
    print(f'\nUpdated {CREDS_FILE} — {"tokens present" if has_token else "no tokens yet, Section 2 will authorize"}')

Loaded credentials from .env
  Client_ID     : dj0yJmk9Snlk...
  Client_Secret : 0e92a00b...
  Yahoo_Email   : michaelesands@gmail.com
  Yahoo_Password: (set)

Updated browser/yahoo_oauth.json — tokens present


## Section 2: OAuth Authorization (Automated)

Uses Playwright headless Chromium to complete the Yahoo OAuth2 flow automatically:
1. Opens the Yahoo authorization URL in a headless browser
2. Logs in with `Yahoo_Email` / `Yahoo_Password` from `.env`
3. Clicks **Agree** on the permissions consent page
4. Extracts the authorization code and exchanges it for access tokens
5. Saves tokens to `yahoo_oauth.json` for future runs

**Subsequent runs:** existing tokens are reused (and auto-refreshed when near expiry) — no browser launch needed.

In [61]:
import asyncio
import base64
import threading
import time
from urllib.parse import urlencode

import requests
from yahoo_oauth import OAuth2

# yahoo_oauth sets its own logger to DEBUG on import — suppress it here
import logging
logging.getLogger('yahoo_oauth').setLevel(logging.WARNING)

STATE_FILE = 'browser/yahoo_browser_state.json'


def _has_refresh_token(creds_file):
    """Return True if a refresh_token exists (can renew access without browser)."""
    if not os.path.exists(creds_file):
        return False
    with open(creds_file) as f:
        data = json.load(f)
    return bool(data.get('refresh_token'))


def _is_2fa_page(url):
    """True only for the real 2FA selector, not the password-entry page."""
    return 'challenge-selector' in url or 'challenge/verify' in url


def _get_oauth_code(consumer_key, email, password):
    """
    Run Playwright in a separate thread (own event loop) to avoid
    conflicting with Jupyter's running asyncio loop.

    Tries saved browser state first (headless=True — may skip 2FA).
    Falls back to visible headless=False browser so the user can approve
    the iPhone push notification once if Yahoo still requires it.
    """
    auth_url = 'https://api.login.yahoo.com/oauth2/request_auth?' + urlencode({
        'client_id':     consumer_key,
        'redirect_uri':  'oob',
        'response_type': 'code',
    })
    result = {'verifier': None, 'error': None}

    def _task():
        asyncio.set_event_loop(asyncio.new_event_loop())
        try:
            from playwright.sync_api import sync_playwright
            with sync_playwright() as pw:
                configs = []
                if os.path.exists(STATE_FILE):
                    configs.append({'headless': True,  'state': STATE_FILE})
                configs.append({'headless': False, 'state': None})

                for cfg in configs:
                    print(f'  Browser: headless={cfg["headless"]}  '
                          f'saved_state={cfg["state"] is not None}')
                    browser = pw.chromium.launch(
                        headless=cfg['headless'],
                        args=['--disable-blink-features=AutomationControlled'],
                    )
                    ctx = browser.new_context(
                        storage_state=cfg['state'],
                        user_agent=(
                            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                            'AppleWebKit/537.36 (KHTML, like Gecko) '
                            'Chrome/122.0.0.0 Safari/537.36'
                        ),
                        locale='en-US',
                    )
                    page = ctx.new_page()

                    try:
                        page.goto(auth_url, wait_until='domcontentloaded', timeout=30000)
                        page.wait_for_timeout(2000)

                        # ── Email ──────────────────────────────────────────────
                        if page.query_selector('#login-username'):
                            page.fill('#login-username', email)
                            page.keyboard.press('Enter')
                            page.wait_for_url(
                                lambda u: 'challenge/password' in u or 'consent' in u or _is_2fa_page(u),
                                timeout=15000,
                            )
                            page.wait_for_timeout(1500)

                        # ── Password ───────────────────────────────────────────
                        if page.query_selector('#login-passwd'):
                            page.fill('#login-passwd', password)
                            page.keyboard.press('Enter')
                            page.wait_for_url(
                                lambda u: 'challenge/password' not in u,
                                timeout=15000,
                            )
                            page.wait_for_timeout(2000)

                        # ── 2FA challenge ──────────────────────────────────────
                        if _is_2fa_page(page.url):
                            if cfg['headless']:
                                print('  Headless hit 2FA — retrying with visible browser')
                                browser.close()
                                continue

                            iphone_btn = page.query_selector('button[name="index"][value="1"]')
                            if iphone_btn:
                                iphone_btn.click()
                                print('  iPhone push sent — approve on your device...')
                            else:
                                print('  2FA required — complete it in the browser window...')

                            for _ in range(300):
                                if not _is_2fa_page(page.url): break
                                time.sleep(1)
                            else:
                                raise RuntimeError('2FA timed out after 5 minutes')
                            print('  2FA cleared')

                        # ── Consent (Agree button starts disabled) ─────────────
                        if 'consent' in page.url or 'authorize' in page.url:
                            page.evaluate('window.scrollTo(0, document.body.scrollHeight)')
                            page.wait_for_timeout(1500)
                            page.evaluate("""() => {
                                const b = document.querySelector('button[name="agree"]')
                                       || document.querySelector('input[name="agree"]')
                                       || document.querySelector('[value="agree"]');
                                if (b) { b.removeAttribute('disabled'); b.click(); }
                            }""")
                            page.wait_for_timeout(3000)

                        # ── Extract verifier ───────────────────────────────────
                        page.wait_for_load_state('networkidle', timeout=15000)

                        verifier = None
                        for sel in ['input[id*="code"]', 'input[id*="oauth"]', 'input[readonly]',
                                    '.auth-code', '#oauth-code', 'code', 'pre']:
                            el = page.query_selector(sel)
                            if el:
                                val = (el.get_attribute('value') or el.inner_text() or '').strip()
                                if len(val) >= 6: verifier = val; break

                        if not verifier:
                            html = page.content()
                            for pat in [
                                r'<code[^>]*>\s*([A-Za-z0-9/_\-]{6,})\s*</code>',
                                r'<pre[^>]*>\s*([A-Za-z0-9/_\-]{6,})\s*</pre>',
                                r'(?:code|verifier)[^<]{0,40}?([A-Za-z0-9/_\-]{8,})',
                            ]:
                                m = re.search(pat, html, re.IGNORECASE)
                                if m: verifier = m.group(1).strip('"\''); break

                        if verifier:
                            ctx.storage_state(path=STATE_FILE)
                            result['verifier'] = verifier
                            browser.close()
                            return
                        else:
                            page.screenshot(path='browser/yahoo_auth_debug.png')
                            raise RuntimeError(
                                'Could not extract OAuth code — screenshot saved to browser/yahoo_auth_debug.png'
                            )

                    except Exception as inner_e:
                        if cfg['headless']:
                            print(f'  Headless failed: {inner_e}')
                            try: browser.close()
                            except: pass
                            continue
                        raise
                    finally:
                        if not result['verifier']:
                            try: browser.close()
                            except: pass

        except Exception as e:
            import traceback; traceback.print_exc()
            result['error'] = e

    t = threading.Thread(target=_task, daemon=True)
    t.start()
    t.join(timeout=400)

    if t.is_alive():
        raise RuntimeError('Auth timed out after 400 seconds')
    if result['error']:
        raise result['error']
    return result['verifier']


def _yahoo_auto_auth(consumer_key, consumer_secret, email, password, creds_file):
    print('Launching browser for Yahoo OAuth...')
    verifier = _get_oauth_code(consumer_key, email, password)
    print(f'Authorization code obtained.')

    b64 = base64.b64encode(f'{consumer_key}:{consumer_secret}'.encode()).decode()
    resp = requests.post(
        'https://api.login.yahoo.com/oauth2/get_token',
        headers={'Authorization': f'Basic {b64}',
                 'Content-Type': 'application/x-www-form-urlencoded'},
        data={'grant_type': 'authorization_code', 'code': verifier, 'redirect_uri': 'oob'},
        timeout=30,
    )
    resp.raise_for_status()
    tokens = resp.json()

    with open(creds_file) as f:
        existing = json.load(f)
    existing.update({
        'access_token':      tokens['access_token'],
        'refresh_token':     tokens.get('refresh_token', ''),
        'token_type':        tokens.get('token_type', 'bearer'),
        'token_expires_in':  tokens.get('expires_in', 3600),
        'token_time':        time.time(),
        'xoauth_yahoo_guid': tokens.get('xoauth_yahoo_guid', ''),
    })
    with open(creds_file, 'w') as f:
        json.dump(existing, f, indent=2)
    print(f'Tokens saved to {creds_file}.')


# ── Main auth logic ───────────────────────────────────────────────────────────
if _has_refresh_token(CREDS_FILE):
    print('Refresh token found — no browser needed.')
else:
    _yahoo_auto_auth(CONSUMER_KEY, CONSUMER_SECRET, YAHOO_EMAIL, YAHOO_PASSWORD, CREDS_FILE)

oauth = OAuth2(None, None, from_file=CREDS_FILE)
if not oauth.token_is_valid():
    oauth.refresh_access_token()

print('OAuth OK — session ready')

Refresh token found — no browser needed.
OAuth OK — session ready


## Section 3: Find Your League ID

Lists all your active MLB fantasy leagues. Copy the `league_id` for your team into the config cell below.

In [62]:
from datetime import date

SEASON = 2026
BASE   = 'https://fantasysports.yahooapis.com/fantasy/v2'

_SCORING_LABEL = {
    'head':      'categories',
    'headpoint': 'points',
    'roto':      'roto',
}

def _get(path):
    if not oauth.token_is_valid():
        oauth.refresh_access_token()
    resp = oauth.session.get(f'{BASE}{path}?format=json')
    resp.raise_for_status()
    return resp.json()

def _flat(data):
    """Collapse Yahoo's list-of-single-key-dicts into one dict."""
    if isinstance(data, dict):
        return data
    out = {}
    for item in data:
        if isinstance(item, dict):
            out.update(item)
    return out

def _my_team_name(league_key):
    """Return the name of the user's team in a given league."""
    data = _get(f'/league/{league_key}/teams')
    teams = data['fantasy_content']['league'][1]['teams']
    for i in range(teams['count']):
        t = _flat(teams[str(i)]['team'][0])
        if t.get('is_owned_by_current_login') == 1:
            return t.get('name', '')
    return ''

# Fetch all MLB leagues for the authenticated user
data = _get(f'/users;use_login=1/games;game_codes=mlb;seasons={SEASON}/leagues')

users = data['fantasy_content']['users']
games = users['0']['user'][1]['games']
leagues_raw = games['0']['game'][1]['leagues']

rows = []
for i in range(leagues_raw['count']):
    lg = _flat(leagues_raw[str(i)]['league'][0])
    league_key   = lg.get('league_key', '')
    scoring_type = lg.get('scoring_type', '')
    rows.append({
        'league_id':      lg.get('league_id', ''),
        'team_name':      _my_team_name(league_key),
        'name':           lg.get('name', ''),
        'scoring_method': _SCORING_LABEL.get(scoring_type, scoring_type),
        'num_teams':      lg.get('num_teams', ''),
        'draft_status':   lg.get('draft_status', ''),
        'league_key':     league_key,
    })

leagues_df = pd.DataFrame(rows)
print(f'Found {len(leagues_df)} MLB league(s) for {SEASON}:')
leagues_df

Found 8 MLB league(s) for 2026:


,league_id,team_name,name,scoring_method,num_teams,draft_status,league_key
0,18397,Mother Tuckers 100,Yahoo Prize H2H-Cat 18397,categories,12,postdraft,469.l.18397
1,41036,Dr. StrangeGlove 100,Yahoo Prize H2H-Cat 41036,categories,12,postdraft,469.l.41036
2,15981,Suave 100,Yahoo Prize Roto 15981,roto,12,postdraft,469.l.15981
3,68331,Japanese Kokamo Okamo 100,Yahoo Prize H2H-Cat 68331,categories,12,postdraft,469.l.68331
4,148341,Schlittler 100,Yahoo Prize H2H-Cat 148341,categories,12,postdraft,469.l.148341
5,3924,My Bradish Burns 100,Yahoo Prize H2H-Cat 3924,categories,12,postdraft,469.l.3924
6,690,Big Dick 000,Yahoo H2H-Cat 690,categories,10,postdraft,469.l.690
7,156186,Big League Richard,Yahoo Prize Roto 156186,roto,12,predraft,469.l.156186


## Section 4: Config — set your league ID

In [63]:
# ── Set this from the output above ───────────────────────────────────────────
LEAGUE_ID = leagues_df['league_id'].iloc[0]   # auto-picks first league; change if you have multiple
# LEAGUE_ID = '12345'                          # or hardcode here
# ─────────────────────────────────────────────────────────────────────────────

print(f'League: {leagues_df.loc[leagues_df.league_id == LEAGUE_ID, "name"].iloc[0]}')
print(f'ID:     {LEAGUE_ID}')

League: Yahoo Prize H2H-Cat 18397
ID:     18397


## Section 5: Initialize Yahoo Client

In [64]:
import importlib
import yahoo as _yahoo_mod
importlib.reload(_yahoo_mod)
from yahoo import Yahoo

yf = Yahoo(league_id=LEAGUE_ID, season=SEASON, creds_file=CREDS_FILE).fetch()

print(yf)

Yahoo(league=469.l.18397, team=469.l.18397.t.9)


## Section 6: Your Team

In [65]:
# Current roster
roster = yf.roster
print(f'Roster ({len(roster)} players):')
roster

Roster (23 players):


,slot,name,team,positions,status,player_key
0,1B,Rafael Devers,SF,1B,DTD,469.p.10235
1,2B,Xavier Edwards,MIA,"2B,SS",,469.p.11375
2,3B,Matt Chapman,SF,3B,,469.p.10205
3,BN,Victor Scott II,STL,OF,,469.p.61888
4,BN,Michael Wacha,KC,SP,,469.p.9329
5,BN,Andrew Painter,PHI,SP,,469.p.60592
6,BN,Sean Manaea,NYM,"SP,RP",,469.p.9582
7,BN,Will Warren,NYY,SP,,469.p.60295
8,C,Yainer Diaz,HOU,C,,469.p.12435
9,OF,Kyle Tucker,LAD,OF,,469.p.10480


In [66]:
# Current matchup
m = yf.matchup
if m:
    print(f"Week {m['week']}  ({m['start']} → {m['end']})")
    print(f"  You:      {m.get('you', '—')}")
    print(f"  Opponent: {m.get('opponent', '—')}")
else:
    print('No active matchup found (offseason or between weeks)')

Week 1  (2026-03-25 → 2026-03-29)
  You:      Mother Tuckers 100
  Opponent: $100 Standard


In [67]:
# Opponent's roster
if m:
    opp_roster = yf.opponent_roster
    print(f"Opponent roster — {m.get('opponent', '—')} ({len(opp_roster)} players):")
    display(opp_roster)
else:
    print('No active matchup.')

Opponent roster — $100 Standard (23 players):


,slot,name,team,positions,status,player_key
0,1B,Vinnie Pasquantino,KC,1B,,469.p.12449
1,2B,Jose Altuve,HOU,"2B,OF",,469.p.8996
2,3B,Manny Machado,SD,3B,,469.p.9111
3,BN,Noelvi Marte,CIN,"3B,OF",,469.p.11529
4,BN,Ezequiel Tovar,COL,SS,,469.p.12322
5,BN,Brenton Doyle,COL,OF,DTD,469.p.12136
6,BN,Kyle Manzardo,CLE,1B,,469.p.60411
7,BN,Joe Musgrove,SD,SP,,469.p.10185
8,C,Gabriel Moreno,AZ,C,,469.p.11855
9,OF,Fernando Tatis Jr.,SD,OF,,469.p.10639


## Section 7: League Standings

In [68]:
yf.standings

,rank,team,wins,losses,ties,pct,gb
0,1,Cruz’s Cool Crew,0,0,0,0.0,-
1,2,Underdog,0,0,0,0.0,-
2,3,Blizzard 💯 ⚾️,0,0,0,0.0,-
3,4,New York Yankees,0,0,0,0.0,-
4,5,$100 Standard,0,0,0,0.0,-
5,6,Don't Know How to Close,0,0,0,0.0,-
6,7,Couch Hundy,0,0,0,0.0,-
7,8,Henchmen,0,0,0,0.0,-
8,9,Mother Tuckers 100,0,0,0,0.0,-
9,10,Bronx Bombers,0,0,0,0.0,-


## Section 8: Waiver Wire & Free Agents

In [69]:
# All players on waivers (paginated — fetches every player)
waivers = yf.waivers()
print(f'Waiver wire ({len(waivers)} players):')
waivers

Waiver wire (2068 players):


,name,team,positions,status,player_key
0,Jonathan Aranda,TB,1B,,469.p.12309
1,Masyn Winn,STL,SS,,469.p.12025
2,Andrew Vaughn,MIL,1B,,469.p.11866
3,Trent Grisham,NYY,OF,,469.p.10522
4,Daulton Varsho,TOR,OF,,469.p.10843
...,...,...,...,...,...
2063,Maverick Handley,BAL,C,NA,469.p.60901
2064,Bradley Blalock,MIA,SP,,469.p.63311
2065,Jack Kochanowicz,LAA,SP,,469.p.11774
2066,Germán Márquez,SD,SP,,469.p.10402


In [70]:
# All free agents (paginated — fetches every player)
# Note: many waiver-only leagues return 0 free agents; all players are on waivers instead
fa = yf.free_agents()
print(f'Free agents: {len(fa)} player(s)')
fa

Free agents: 0 player(s)


,name,team,positions,status,player_key


In [71]:
# Filter by position — change 'SP' to any Yahoo position code
# Positions: C  1B  2B  3B  SS  OF  Util  SP  RP  P
waivers_sp = yf.waivers(position='SP')
print(f'SP on waivers ({len(waivers_sp)}):')
waivers_sp

SP on waivers (637):


,name,team,positions,status,player_key
0,José Soriano,LAA,SP,DTD,469.p.11327
1,Reynaldo López,ATL,SP,,469.p.10152
2,Bailey Ober,MIN,SP,,469.p.12096
3,Mitch Keller,PIT,SP,,469.p.10565
4,Yusei Kikuchi,LAA,SP,,469.p.11263
...,...,...,...,...,...
632,Cal Quantrill,TEX,SP,NA,469.p.10573
633,Bradley Blalock,MIA,SP,,469.p.63311
634,Jack Kochanowicz,LAA,SP,,469.p.11774
635,Germán Márquez,SD,SP,,469.p.10402


In [72]:
# Search for a specific player (any ownership status)
yf.search('Pete Alonso')

,name,team,positions,status,player_key
0,Pete Alonso,BAL,1B,,469.p.10918


## Section 9: Add / Drop Players

Uncomment and edit the player names to execute transactions.

In [73]:
# Add a free agent
# yf.add('Pete Alonso')

# Add and simultaneously drop another player
# yf.add('Pete Alonso', drop='Joey Votto')

# FAAB leagues: include a bid amount
# yf.add('Pete Alonso', drop='Joey Votto', faab=15)

# Drop only
# yf.drop('Joey Votto')

print('Uncomment a line above to execute a transaction.')

Uncomment a line above to execute a transaction.


## Section 10: Lineup Management

In [74]:
# Quick-reference: your bench and IL slots right now
bench = roster[roster['slot'].isin(['BN', 'IL', 'IL+', 'NA'])]
print('Bench / IL:')
print(bench[['slot', 'name', 'team', 'positions', 'status']].to_string(index=False))

Bench / IL:
slot            name team positions status
  BN Victor Scott II  STL        OF       
  BN   Michael Wacha   KC        SP       
  BN  Andrew Painter  PHI        SP       
  BN     Sean Manaea  NYM     SP,RP       
  BN     Will Warren  NYY        SP       


In [75]:
# Move a player to a specific slot
# yf.move('Yainer Diaz', 'BN')
# yf.move('Aaron Judge', 'BN')
# yf.move('Jacob deGrom', 'IL')

# Bench a starter
# yf.bench('Cody Bellinger')

# Move a player to IL
# yf.il('Spencer Strider')

# Activate a bench player (moves to first open eligible slot)
# yf.start('Cody Bellinger')

# Swap two players' slots (e.g. start one, bench the other)
# yf.swap('Bench Player', 'Active Player')

print('Uncomment a line above to change your lineup.')

Uncomment a line above to change your lineup.


In [76]:
# Confirm updated roster after changes
# yf.roster

## Section 11: Trade Proposals

Search for players on other teams before proposing. The `team` argument matches any substring of the opponent's team name.

In [77]:
# Search for a player to check their current team / ownership
yf.search('Julio Rodriguez')

,name,team,positions,status,player_key
0,Julio Rodríguez,SEA,OF,,469.p.11384


In [78]:
# Propose a trade
# yf.trade(
#     give=['Your Player Name'],
#     receive=['Their Player Name'],
#     team='Opponent Team Name',     # case-insensitive substring of their team name
# )

# Multi-player trade
# yf.trade(
#     give=['Player A', 'Player B'],
#     receive=['Player C'],
#     team='Rival Squad',
# )

print('Uncomment above to propose a trade.')

Uncomment above to propose a trade.


## Section 12: Top Available Players Across All Leagues

Fetches the top available batters and pitchers by % rostered for each of your leagues.
Set `N_PLAYERS` to control how many of each type are returned per league.
Players are pulled from waivers (`status=W`) sorted by percent owned descending.

In [79]:
N_PLAYERS = 3   # top batters + top pitchers to return per league

_PITCHER_POS = {'SP', 'RP', 'P'}

def _is_pitcher(positions: str) -> bool:
    return bool(set(positions.split(',')) & _PITCHER_POS)

def _parse_pct_owned(p1) -> float:
    """
    Extract percent_owned value from player[1].
    Yahoo returns percent_owned as a list-of-single-key-dicts:
      [{'coverage_type': 'week', 'week': 1}, {'value': 42}, {'delta': '0'}]
    Use _flat() to collapse, then read 'value'.
    """
    if not isinstance(p1, dict):
        return 0.0
    po = p1.get('percent_owned', [])
    if isinstance(po, list):
        return float(_flat(po).get('value', 0))
    if isinstance(po, dict):
        return float(po.get('value', 0))
    return float(po)

def _fetch_top_available(league_key: str, n: int = 100) -> pd.DataFrame:
    """
    Fetch available waiver players sorted by % rostered (descending).
    Fetches n players by average rank, then re-sorts by pct_owned in Python.
    """
    data = _get(f'/league/{league_key}/players;status=W;count={n};out=percent_owned;sort=AR')
    players = data['fantasy_content']['league'][1]['players']
    if not isinstance(players, dict):
        return pd.DataFrame(columns=['name', 'team', 'positions', 'status', 'pct_owned'])

    rows = []
    for i in range(players.get('count', 0)):
        p = players[str(i)]['player']
        flat = _flat(p[0])
        name_raw = flat.get('name', '')
        name = name_raw.get('full', '') if isinstance(name_raw, dict) else str(name_raw)
        rows.append({
            'name':      name,
            'team':      flat.get('editorial_team_abbr', ''),
            'positions': flat.get('display_position', ''),
            'status':    flat.get('status', ''),
            'pct_owned': _parse_pct_owned(p[1] if len(p) > 1 else {}),
        })

    return (
        pd.DataFrame(rows)
        .sort_values('pct_owned', ascending=False)
        .reset_index(drop=True)
    )


# ── Loop through all leagues ──────────────────────────────────────────────────
results = []

for _, row in leagues_df.iterrows():
    league_key  = row['league_key']
    league_name = row['name']
    team_name   = row['team_name']
    print(f'  {league_name}...')

    try:
        df = _fetch_top_available(league_key, n=100)

        batters  = df[~df['positions'].apply(_is_pitcher)].head(N_PLAYERS)
        pitchers = df[ df['positions'].apply(_is_pitcher)].head(N_PLAYERS)

        for rank, (_, p) in enumerate(batters.iterrows(),  1):
            results.append({'league': league_name, 'my_team': team_name,
                            'type': 'batter',  'rank': rank, **p})
        for rank, (_, p) in enumerate(pitchers.iterrows(), 1):
            results.append({'league': league_name, 'my_team': team_name,
                            'type': 'pitcher', 'rank': rank, **p})
    except Exception as e:
        print(f'    Error: {e}')

top_available = (
    pd.DataFrame(results)
    [['league', 'my_team', 'type', 'rank', 'name', 'team', 'positions', 'status', 'pct_owned']]
)

print(f'\nDone — {len(top_available)} rows ({len(leagues_df)} leagues × 2 types × {N_PLAYERS} players)')
top_available

  Yahoo Prize H2H-Cat 18397...
  Yahoo Prize H2H-Cat 41036...
  Yahoo Prize Roto 15981...
  Yahoo Prize H2H-Cat 68331...
  Yahoo Prize H2H-Cat 148341...
  Yahoo Prize H2H-Cat 3924...
  Yahoo H2H-Cat 690...
  Yahoo Prize Roto 156186...

Done — 48 rows (8 leagues × 2 types × 3 players)


,league,my_team,type,rank,name,team,positions,status,pct_owned
0,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,1,Jonathan Aranda,TB,1B,,73.0
1,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,2,Masyn Winn,STL,SS,,55.0
2,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,3,Trent Grisham,NYY,OF,,50.0
3,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,pitcher,1,Yusei Kikuchi,LAA,SP,,22.0
4,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,pitcher,2,Mitch Keller,PIT,SP,,14.0
5,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,pitcher,3,Edwin Uceta,TB,RP,DTD,14.0
6,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,1,Luis García Jr.,WSH,2B,,71.0
7,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,2,Kerry Carpenter,DET,OF,,63.0
8,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,3,Bryson Stott,PHI,"2B,SS",,56.0
9,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,pitcher,1,Sean Manaea,NYM,"SP,RP",,30.0
